# Synth-Artiste Showdown

**Research question:** Do two explicitly specified simulated styles produce distinguishable image portfolios when subjects and generation providers are matched?

**Operational measures:** Within- and between-artist distances, cross-artist nearest-neighbor frequency, and agreement among DreamSim, LPIPS, and style-conditioned TPIPS. We visualize the same images in global and neighborhood-focused projections.

This is a small descriptive experiment, not an evaluation of real artists, creativity, or human similarity judgments. Both artists use both GPT-Image 2.5 and Nano Banana 2. Three shared subjects per artist and provider yield 12 selected images. Two candidates per subject plus four anchors require **28 paid image generations** on a fresh run. Exact cached requests are reused. Increase replication for scientific claims; API generation is not seed-reproducible.

Install `.[vision,notebooks]` and configure the two provider keys. Run on CUDA for metric evaluation. This notebook is self-contained and saves all candidates and provenance.

In [ ]:
%matplotlib inline
from pathlib import Path
import os
from dotenv import load_dotenv
ROOT = Path.cwd() if (Path.cwd() / "synthart").exists() else Path.cwd().parent
load_dotenv(ROOT / ".env.local")
import numpy as np
import matplotlib.pyplot as plt
from synthart import ImageGenerator, Similarity, FlowMapGenerator, generate_similar
from synthart.images import release_memory
from synthart.plotting import gallery, plot_spaces
from synthart.experiment import cached_image
OUT = ROOT / "outputs" / "notebook-tour"
OUT.mkdir(parents=True, exist_ok=True)


## Matched Portfolios and Similarity Conditioning

The two simulated artists differ in explicit visual instructions. Their anchor subject is held out from the evaluation subjects. DreamSim selects the closer of two text-generated candidates for each subject. Both the first-candidate baseline and the selected image are retained, allowing the selection effect to be inspected. This design does not pretend that DreamSim embeddings directly condition the provider's hidden model.

In [ ]:
from synthart.experiment import ARTISTS, SUBJECTS, make_portfolios, evaluate_portfolios
RUN = ROOT / "outputs" / "showdown"
manifest = make_portfolios(RUN, backends=("openai", "gemini"), candidates=2)
records = manifest["records"]
print({"images": len(records), "subjects": SUBJECTS, "artists": [a.name for a in ARTISTS]})
assert len(records) == 12
paths = [RUN / r["path"] for r in records]
labels = [r["artist"] for r in records]
gallery(paths, [f'{i}: {r["artist"]} / {r["backend"]}' for i,r in enumerate(records)], columns=4)
plt.show()

## Evaluate in the Original Spaces

DreamSim is the selection metric, so its results are not an independent validation. LPIPS and TPIPS provide held-out model perspectives, although all three have learned biases. The summary is computed in each original space, with separate results for each provider to expose provider effects. A large between-artist distance can reflect palette or subject treatment rather than a distinct artistic identity.

In [ ]:
report = evaluate_portfolios(RUN)
import json
print(json.dumps(report, indent=2))
rdms = {name: np.load(RUN / f"{name}-distances.npy") for name in report}
selection_gains = [r["distances"][0] - min(r["distances"]) for r in records]
print("Mean reference-distance decrease from selecting candidates:", float(np.mean(selection_gains)))
print("Fraction selecting a candidate other than baseline:", float(np.mean([r["selected"] != 0 for r in records])))

## Global, Local, and Multiscale Views

Metric MDS minimizes distance distortion and targets global pairwise geometry. t-SNE emphasizes local neighborhoods; distances between islands and their apparent areas are not reliable overlap estimates. UMAP builds a neighborhood graph at a chosen scale and is useful for multiscale exploration, but it does not guarantee global distance preservation.

Each point is one generated image, numbered as in the gallery. Color denotes artist. A global rank-correlation diagnostic accompanies every projection. With only 12 images, all layouts are unstable; treat overlap as a property to measure in the original space, not something proved by a 2D cluster.

In [ ]:
projection_diagnostics = {}
for name, distances in rdms.items():
    fig, diagnostics = plot_spaces(distances, labels)
    fig.suptitle(f"{name}: Same Images, Different Geometric Objectives", y=1.03)
    plt.show()
    projection_diagnostics[name] = diagnostics
print(json.dumps(projection_diagnostics, indent=2))

## Matched-Subject Comparison and Interpretation

The comparison below holds provider and subject fixed when pairing artists. Compare it with the pooled between-artist result: pooled distances also include subject differences. The small sample does not support significance claims. In a larger experiment, vary seeds or repeated API samples, preregister the TPIPS factor, include human judgments, and evaluate on subjects not used to tune prompts.

In [ ]:
for name, distances in rdms.items():
    pairs = [(i,j) for i,a in enumerate(records) for j,b in enumerate(records)
             if i < j and a["artist"] != b["artist"] and a["backend"] == b["backend"] and a["subject"] == b["subject"]]
    print(name, "matched-subject between-artist distance:", float(np.mean([distances[i,j] for i,j in pairs])))
# Exercise scaffold: rerun project(..., seed=7) and compare neighbors, not plot orientation.
from synthart import project
xy, diagnostic = project(rdms["dreamsim"], "mds", seed=7)
print("Alternative initialization:", diagnostic)

## Limitations and Further Reading

The experiment uses two designed prompt profiles and a small, deliberately matched subject set. Style, palette, composition, and semantics remain entangled. Best-of-N selection improves its own objective by construction, and the reference itself comes from the generator. Provider updates can change results even when model aliases are unchanged. Preserve the manifests and saved images when reporting results.

See [the literature guide](../docs/literature.md) for DreamSim, LPIPS, TPIPS, representational similarity analysis, MDS, t-SNE, UMAP, and related work connecting cognitive science and visual representations.